# Module 2 — Dynamic Topology Extraction

Captures MLP activations, builds Pair Representations and Universal Modules, and exports the dynamic feature atlas.

In [ ]:
# Cell 1 – Dependency setup (circuit_sparsity injection + pip installs)
# version 1.20
import subprocess, sys, types, importlib.util

# ── pip installs ─────────────────────────────────────────────────────────
pkgs = ["h5py", "umap-learn", "seaborn", "transformers", "torch",
        "numpy", "pandas", "tqdm", "pyarrow", "huggingface_hub"]
for pkg in pkgs:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)

# ── Download real gpt.py & hook_utils.py from HuggingFace ───────────────
from huggingface_hub import hf_hub_download

REPO_ID = "openai/circuit-sparsity"
gpt_path = hf_hub_download(repo_id=REPO_ID, filename="gpt.py")
hook_path = hf_hub_download(repo_id=REPO_ID, filename="hook_utils.py")

# ── Load them as circuit_sparsity.gpt and circuit_sparsity.hook_utils ───
def _load_module(mod_name, file_path):
    spec = importlib.util.spec_from_file_location(mod_name, file_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)
    return mod

cs_pkg = types.ModuleType("circuit_sparsity")
cs_pkg.__path__ = []
sys.modules["circuit_sparsity"] = cs_pkg

gp = _load_module("circuit_sparsity.gpt", gpt_path)
hu = _load_module("circuit_sparsity.hook_utils", hook_path)
cs_pkg.gpt = gp
cs_pkg.hook_utils = hu

print("circuit_sparsity loaded from real HF files (v1.20)")
print(f"  gpt.py       : {gpt_path}")
print(f"  hook_utils.py: {hook_path}")
print(f"  GPTConfig    : {hasattr(gp, 'GPTConfig')}")

In [ ]:
# Cell 2 – Configuration
# version 1.03

MODEL_ID       = "openai/circuit-sparsity"
#PARQUET_PATH   = "/content/drive/MyDrive/DATA/CSP-Atlas/small_40x50x50_validated_prompts.parquet"
PARQUET_PATH   = "/content/drive/MyDrive/DATA/CSP-Atlas/test_5x5x10_validated_prompts.parquet"
CHECKPOINT_DIR = "/content/drive/MyDrive/DATA/CSP-Atlas/checkpoints"
OUTPUT_HDF5    = "/content/drive/MyDrive/DATA/CSP-Atlas/dynamic_feature_atlas.h5"
STATS_JSON     = "/content/drive/MyDrive/DATA/CSP-Atlas/extraction_stats.json"

N_LAYERS            = 8        # circuitgpt has 8 transformer blocks
EPSILON             = 1e-3     # soft-to-hard binarization threshold
CONSISTENCY_THRESH  = 0.8      # fraction of variations neuron must fire in
CHECKPOINT_EVERY    = 200      # save checkpoint every N pairs
RESUME_CKPT         = None     # set to path string to resume

import os
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
print("Config OK")

In [ ]:
# Cell 3 – Imports & path setup
# On Colab: run the cell below to clone the repo first (if not already done):
#   !git clone https://github.com/piotrwilam/CSP-Atlas.git /content/CSP-Atlas

import os, sys, logging
import numpy as np
import pandas as pd
import torch
import h5py
from tqdm.auto import tqdm

# ── sys.path: local dev or Colab ─────────────────────────────────────────
LOCAL_SRC = "/Users/piotrwilam/CODE/CSP-Atlas/src"
COLAB_SRC = "/content/CSP-Atlas/src"
SRC_PATH  = LOCAL_SRC if os.path.isdir(LOCAL_SRC) else COLAB_SRC
if SRC_PATH not in sys.path:
    sys.path.insert(0, SRC_PATH)

logging.basicConfig(level=logging.INFO,
                    format="%(asctime)s %(levelname)s %(name)s: %(message)s")
logger = logging.getLogger("module2")
print("Imports OK | torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
print(f"src path : {SRC_PATH}  (exists: {os.path.isdir(SRC_PATH)})")


In [ ]:
# Cell 4 – Mount Drive & load data
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import pandas as pd
df = pd.read_parquet(PARQUET_PATH)
print(f"Loaded {len(df)} rows | "
      f"{df.groupby(['ast_node','builtin_obj']).ngroups} unique pairs")
df.head(3)


In [ ]:
# Cell 5 – Load model & tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    dtype=torch.float32,
).to(DEVICE).eval()

print(f"Model loaded on {DEVICE}")
print(f"Params: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
# Cell 6 – ActivationExtractor: auto-detect MLP layers + register hooks
import re
from module2.extraction import ActivationExtractor

# ── Auto-detect MLP hook pattern from model architecture ─────────────────
print("Scanning model.named_modules() for MLP layers...")
mlp_pattern = None
seen_lids    = set()

for name, _ in model.named_modules():
    # Match names ending in a digit-indexed .mlp  e.g. "blocks.0.mlp"
    m = re.match(r'^(.*?)(\d+)(\.mlp)$', name)
    if m:
        lid = int(m.group(2))
        if lid not in seen_lids:
            seen_lids.add(lid)
            print(f"  [{lid}] {name}")
        if mlp_pattern is None:
            mlp_pattern = m.group(1) + "{layer_id}" + m.group(3)

if mlp_pattern is None:
    # Fallback: dump all module names so you can set mlp_pattern manually
    print("\nCould not auto-detect. All named modules:")
    for name, _ in model.named_modules():
        print(f"  {name}")
    raise RuntimeError(
        "Set mlp_pattern manually (e.g. 'blocks.{layer_id}.mlp') and re-run."
    )

print(f"\nDetected pattern : {mlp_pattern}")
print(f"Layers found     : {sorted(seen_lids)}")

# ── Build extractor with manual hooks ────────────────────────────────────
extractor = ActivationExtractor(
    model=model,
    tokenizer=tokenizer,
    device=DEVICE,
    n_layers=N_LAYERS,
    use_hook_recorder=False,   # manual hooks — reliable across all versions
)
extractor.set_hook_pattern(mlp_pattern)
extractor.register_hooks()
print("Hooks registered.")

# ── Sanity check: extract one prompt ─────────────────────────────────────
sample_prompt = df["prompt_text"].iloc[0]
acts = extractor.extract(sample_prompt)

if not acts:
    print("WARNING: No activations captured — check mlp_pattern above.")
else:
    print(f"\nExtracted {len(acts)} layers:")
    for lid, vec in sorted(acts.items()):
        print(f"  Layer {lid}: shape={tuple(vec.shape)}, "
              f"active={(vec.abs() > EPSILON).sum().item()} / {vec.numel()}")


In [ ]:
# Cell 7 – PairRepresentationBuilder sanity check
from module2.binarization import PairRepresentationBuilder

builder = PairRepresentationBuilder(
    epsilon=EPSILON,
    consistency_threshold=CONSISTENCY_THRESH,
    n_layers=N_LAYERS,
)

# Test on first pair
grouped     = df.groupby(["ast_node", "builtin_obj"])
first_key   = list(grouped.groups.keys())[0]
sample_prompts = df.loc[grouped.groups[first_key], "prompt_text"].tolist()

pair_masks_sample = builder.build(extractor, sample_prompts[:3])
print(f"Pair {first_key}: {len(pair_masks_sample)} layers")
for lid, m in sorted(pair_masks_sample.items()):
    print(f"  Layer {lid}: circuit_size={m.sum()} / {m.size}")


In [ ]:
# Cell 8 – Full pipeline run (with checkpointing)
from module2.pipeline import Module2Pipeline

pipeline = Module2Pipeline(
    extractor=extractor,
    builder=builder,
    parquet_path=PARQUET_PATH,
    checkpoint_dir=CHECKPOINT_DIR,
    checkpoint_every=CHECKPOINT_EVERY,
)

pair_masks, universal_masks, metrics, stats_df = pipeline.run(
    resume_from_checkpoint=RESUME_CKPT
)

print(f"Pairs extracted  : {len(pair_masks)}")
print(f"Universal AST    : {len(universal_masks['ast'])}")
print(f"Universal Builtin: {len(universal_masks['builtin'])}")
stats_df.head()


In [ ]:
# Cell 9 – Inspect circuit sizes
import matplotlib.pyplot as plt

sizes = stats_df["circuit_size"].values
plt.figure(figsize=(8, 4))
plt.hist(sizes, bins=50, edgecolor="black")
plt.xlabel("Circuit size (# active neurons)")
plt.ylabel("Count")
plt.title("Distribution of Pair Representation circuit sizes")
plt.tight_layout()
plt.show()

print(f"Mean circuit size: {sizes.mean():.1f} ± {sizes.std():.1f}")
print(f"Min: {sizes.min()} | Max: {sizes.max()}")


In [ ]:
# Cell 10 – Universal module stats
rep_layer = 4

ast_sizes    = {n: int(lm[rep_layer].sum()) for n, lm in universal_masks["ast"].items()    if rep_layer in lm}
builtin_sizes = {n: int(lm[rep_layer].sum()) for n, lm in universal_masks["builtin"].items() if rep_layer in lm}

print(f"Universal AST modules at layer {rep_layer}: {len(ast_sizes)}")
print(f"Universal Builtin modules at layer {rep_layer}: {len(builtin_sizes)}")

top_ast = sorted(ast_sizes.items(), key=lambda x: -x[1])[:10]
print("\nTop 10 Universal AST circuits:")
for n, s in top_ast:
    print(f"  {n}: {s} neurons")


In [ ]:
# Cell 11 – Jaccard matrix diagnostics
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if "jaccard_ast_matrix" in metrics:
    mat   = metrics["jaccard_ast_matrix"]
    names = metrics.get("ast_names", [str(i) for i in range(mat.shape[0])])

    fig, ax = plt.subplots(figsize=(min(20, len(names)*0.4+2),
                                    min(20, len(names)*0.4+2)))
    sns.heatmap(mat, ax=ax, vmin=0, vmax=1,
                xticklabels=names if len(names) <= 30 else False,
                yticklabels=names if len(names) <= 30 else False,
                cmap="viridis")
    ax.set_title("Jaccard Similarity — Universal AST Modules (layer 4)")
    plt.tight_layout()
    plt.show()

    off_diag = mat[np.triu_indices_from(mat, k=1)]
    print(f"Mean off-diag Jaccard: {off_diag.mean():.4f}")
    print(f"Max  off-diag Jaccard: {off_diag.max():.4f}")


In [ ]:
# Cell 12 – Entanglement Index spot check
import pandas as pd
from module2.metrics import entanglement_index

ei_results = []
rep_layer  = 4

for (ast_n, blt_o), layers in pair_masks.items():
    if rep_layer not in layers:
        continue
    pm = layers[rep_layer]
    am = universal_masks["ast"].get(ast_n, {}).get(rep_layer)
    bm = universal_masks["builtin"].get(blt_o, {}).get(rep_layer)
    if am is None or bm is None:
        continue
    ei = entanglement_index(pm, am, bm)
    ei_results.append({"ast_node": ast_n, "builtin_obj": blt_o, "E_I": ei})

ei_df = pd.DataFrame(ei_results)
print(f"Entanglement Index — {len(ei_df)} pairs at layer {rep_layer}")
print(ei_df["E_I"].describe())
ei_df.sort_values("E_I", ascending=False).head(10)


In [ ]:
# Cell 13 – Layer-wise circuit evolution
import matplotlib.pyplot as plt
import numpy as np

layer_ids = sorted({lid for lm in pair_masks.values() for lid in lm})

mean_per_layer = []
for lid in layer_ids:
    sizes = [lm[lid].sum() for lm in pair_masks.values() if lid in lm]
    mean_per_layer.append(np.mean(sizes) if sizes else 0)

plt.figure(figsize=(7, 4))
plt.plot(layer_ids, mean_per_layer, marker="o")
plt.xlabel("Layer")
plt.ylabel("Mean circuit size (neurons)")
plt.title("Mean Pair Representation circuit size vs. layer")
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 14 – Save HDF5 atlas
from module2.io_utils import save_atlas_hdf5

metadata = {
    "model_id"           : MODEL_ID,
    "n_layers"           : N_LAYERS,
    "epsilon"            : EPSILON,
    "consistency_thresh" : CONSISTENCY_THRESH,
    "n_pairs"            : len(pair_masks),
    "ast_nodes"          : sorted(set(a for a, _ in pair_masks)),
    "builtin_objs"       : sorted(set(b for _, b in pair_masks)),
}

save_atlas_hdf5(OUTPUT_HDF5, pair_masks, universal_masks, metrics, metadata)
print("Atlas saved:", OUTPUT_HDF5)


In [ ]:
# Cell 15 – Save extraction stats JSON
import json

stats_out = {
    "n_pairs"        : len(pair_masks),
    "n_universal_ast": len(universal_masks["ast"]),
    "n_universal_blt": len(universal_masks["builtin"]),
    "circuit_size_mean": float(stats_df["circuit_size"].mean()),
    "circuit_size_std" : float(stats_df["circuit_size"].std()),
}

with open(STATS_JSON, "w") as fh:
    json.dump(stats_out, fh, indent=2)
print("Stats saved:", STATS_JSON)
print(json.dumps(stats_out, indent=2))


In [ ]:
# Cell 16 – Verify HDF5 round-trip
import h5py

with h5py.File(OUTPUT_HDF5, "r") as f:
    def _walk(name, obj):
        if isinstance(obj, h5py.Dataset):
            print(f"  {name}: {obj.shape} {obj.dtype}")
    f.visititems(_walk)
    print("\nMetadata attrs:", dict(f["metadata"].attrs))


In [ ]:
# Cell 17 – Cleanup hooks & done
extractor.remove_hooks()
print("Hooks removed.")
print("\nModule 2 extraction complete.")
print(f"  Atlas  : {OUTPUT_HDF5}")
print(f"  Stats  : {STATS_JSON}")
print(f"  Pairs  : {len(pair_masks)}")
print(f"  Layers : {N_LAYERS}")
